In [1]:
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [2]:
import sys

!"{sys.executable}" -m pip install awswrangler -q
!"{sys.executable}" -m pip install optbinning -q
!"{sys.executable}" -m pip install lightgbm -q
!"{sys.executable}" -m pip install xgboost --prefer-binary -q
!"{sys.executable}" -m pip install boto3 -q


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#from optbinning import BinningProcess
import shutil
from warnings import simplefilter
simplefilter(action = "ignore") #, category = FutureWarning

pd.set_option('display.max_rows', 500)
from sklearn.preprocessing import LabelEncoder

In [4]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [5]:
import pandas as pd
import awswrangler as wr

# Parámetros
bucket_name = 'ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

# Ruta S3 del parquet
s3_path = f"s3://{bucket_name}/{model_prefix}/DATA_INFERENCIA_PILOTO/base_conscore_202604.parquet"

# Cargar parquet desde S3
df_inference = wr.s3.read_parquet(path=s3_path, boto3_session=session)

print(f"✓ Parquet cargado desde S3")
print(f"  Shape: {df_inference.shape}")
print(f"  Columnas: {df_inference.columns.tolist()[:10]}...")  # primeras 10
display(df_inference.head())

✓ Parquet cargado desde S3
  Shape: (172505, 50)
  Columnas: ['target', 'mto_pas_soles', 'imp_trx_abonosefect_6m', 'imp_trx_cargosefe_6m', 'avg_trx_cargostot_3m', 'cnt_trx_cargostot_3m', 'cnt_trx_abonospromtot_3m', 'rat_trx_abonosefectot_1m', 'rat_trx_abonosefectot_3m', 'rat_trx_abonosefectot_9m']...


,target,mto_pas_soles,imp_trx_abonosefect_6m,imp_trx_cargosefe_6m,avg_trx_cargostot_3m,cnt_trx_cargostot_3m,cnt_trx_abonospromtot_3m,rat_trx_abonosefectot_1m,rat_trx_abonosefectot_3m,rat_trx_abonosefectot_9m,...,ingresos_vs_facturacion,pasivo_vs_ingresos,ratio_egresos_exterior,ratio_ingresos_exterior,key_value,cod_cli,cod_mes,tipo_alerta_n2,trx_riesgo_cliente,score
0,0.0,10071.97,0.0,0.0,0.00,16.0,2.00,0.0,0.0,0.0000,...,0.000000,0.564477,0.0,0.0,D51088353B997B152B8E0A6DD806AC55A1BCF80FB36835...,20071988.0,202604.0,0,SIN_INFO,0.086045
1,0.0,7416.00,0.0,33450.0,4.00,46.0,87.00,0.0,0.0,0.0000,...,0.000000,0.068574,0.0,0.0,27557E7E10258CBD9B884C456F432592C9E19577A233B8...,20628792.0,202604.0,0,SIN_INFO,0.009997
2,0.0,210714.50,6982.0,7000.0,0.33,64.0,1.67,0.5,0.2,0.2000,...,0.000000,0.237728,0.0,0.0,E14BC7992EC19EC51B7C4A9A8EFA85BF37A947241573D7...,21046548.0,202604.0,0,SIN_INFO,0.748831
3,0.0,659.90,33400.0,0.0,0.00,11.0,1.33,0.0,0.0,0.2222,...,0.059132,0.006540,0.0,0.0,E7E69BFA77619BAD5D08AF844023B68F629DC7AA02C052...,21853118.0,202604.0,0,SIN_INFO,0.072348
4,0.0,24.90,1300.0,0.0,0.00,4.0,0.33,0.0,1.0,1.0000,...,0.000000,0.019154,0.0,0.0,48BAB665F07CBF5D1083FB9B377724376E2A4FC2470843...,22126020.0,202604.0,0,SIN_INFO,0.113739


In [6]:
import pandas as pd
import awswrangler as wr

# Parámetros
bucket_name = 'ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PJ/MINORISTA'

# Ruta S3 con partición (awswrangler lee automáticamente la partición)
s3_path = f"s3://{bucket_name}/{model_prefix}/DATA_INFERENCIA_PILOTO/INFERENCIA/"

# Cargar parquet desde S3 (lee todas las particiones o filtra por periodo=202604)
df_inferencia = wr.s3.read_parquet(
    path=s3_path, 
    boto3_session=session,
    partition_filter=lambda x: x["periodo"] == ["202604"]  # Filtra solo periodo 202604
)

print(f"✓ Parquet cargado desde S3")
print(f"  Shape: {df_inferencia.shape}")
print(f"  Columnas: {df_inferencia.columns.tolist()[:10]}...")  # primeras 10
print(f"  Dtypes:\n{df_inferencia.dtypes}")
display(df_inferencia.head())

✓ Parquet cargado desde S3
  Shape: (172506, 67)
  Columnas: ['key_value', 'cod_cli', 'codmes_lag1', 'cod_mes', 'fec_constitucion', 'mto_pas_soles', 'imp_trx_abonosefect_6m', 'imp_trx_cargosefe_6m', 'avg_trx_cargostot_3m', 'max_trx_abonos_3m']...
  Dtypes:
key_value                    string[python]
cod_cli                      string[python]
codmes_lag1                  string[python]
cod_mes                               Int32
fec_constitucion                     object
mto_pas_soles                       float64
imp_trx_abonosefect_6m              float64
imp_trx_cargosefe_6m                float64
avg_trx_cargostot_3m                float64
max_trx_abonos_3m                   float64
cnt_trx_cargostot_3m                  Int32
cnt_trx_abonospromtot_3m            float64
rat_trx_abonosefectot_1m            float64
rat_trx_abonosefectot_3m            float64
rat_trx_abonosefectot_9m            float64
rat_mntcrgsefetot_1m                float64
num_edad_constitucion                 I

,key_value,cod_cli,codmes_lag1,cod_mes,fec_constitucion,mto_pas_soles,imp_trx_abonosefect_6m,imp_trx_cargosefe_6m,avg_trx_cargostot_3m,max_trx_abonos_3m,...,alertas_por_antiguedad,ingresos_vs_facturacion,pasivo_vs_ingresos,ratio_egresos_exterior,ratio_ingresos_exterior,gap_riesgo_pep_lsb,tipo_alerta_n2,trx_riesgo_cliente,target_m,periodo_alerta
0,C996D04B0C345F298F48A33478E11DF6E78ED1D3A4962B...,0021536249,202603,202604,2025-02-24,3075.2500,5400.0,0.0,0.00,5000.00,...,NaN,NaN,0.335726,NaN,NaN,NaN,<NA>,<NA>,0,<NA>
1,A84E0854D16BC29F4A3055388D9E42BD78EB52277F9626...,0016085397,202603,202604,2001-04-11,532.7900,2869.0,75000.0,0.33,766.93,...,NaN,0.003424,0.023796,NaN,NaN,NaN,<NA>,<NA>,0,<NA>
2,1CC46BFA8218FEB5883562FDA664E8FB8D4E410F3046E2...,0020742181,202603,202604,2022-05-27,0.0000,0.0,0.0,0.00,3300.00,...,NaN,NaN,0.000000,NaN,NaN,NaN,<NA>,<NA>,0,<NA>
3,61D9981A8AB8587D6535C30FB44F1793ABCED78F5199C9...,0020353274,202603,202604,2023-05-11,41386.5253,6800.0,0.0,0.00,94195.25,...,NaN,NaN,0.112694,NaN,NaN,NaN,<NA>,<NA>,0,<NA>
4,070C26E8A5A4FCDF5050AA22F336AC5F86EE5E0F13C174...,0015509676,202603,202604,2016-10-12,0.0000,NaN,NaN,0.00,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>


In [7]:
def preprocessing_fn(df: pd.DataFrame) -> pd.DataFrame:
    df = df.fillna(0)

    categorical_columns = [
        "desc_provincia", "cnt_ro_debajo_umbral", "mto_fact_declarado_sunat", "flg_activo_pep",
        "desc_departamento", "cod_ubigeo_cd", "cod_sectorista_id", "cod_ciiu_v4"
    ]

    # Filtrar solo columnas que existen en el DataFrame
    existing_cols = [col for col in categorical_columns if col in df.columns]
    missing_cols = [col for col in categorical_columns if col not in df.columns]
    
    if missing_cols:
        print(f"⚠ Columnas no encontradas: {missing_cols}")
    
    print(f"✓ Procesando {len(existing_cols)} columnas: {existing_cols}")

    # Aplicar to_numeric PRIMERO (convierte strings a NaN si no puede convertir)
    for col in existing_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Ahora convertir a float64 (ya son numéricas o NaN)
    for col in existing_cols:
        df[col] = df[col].astype('float64')

    # Rellenar con la media (todas son float64)
    mean_values = df[existing_cols].mean()
    for col in existing_cols:
        df[col] = df[col].fillna(mean_values[col])

    return df

# Aplicar preprocessing
try:
    df_inferencia_processed = preprocessing_fn(df_inferencia.copy())
    
    print(f"✓ Preprocessing aplicado")
    print(f"  Shape: {df_inferencia_processed.shape}")
    print(f"  Dtypes:\n{df_inferencia_processed.dtypes}")
    display(df_inferencia_processed.head())
except Exception as e:
    print(f"❌ Error en preprocessing: {str(e)}")
    import traceback
    traceback.print_exc()

✓ Procesando 8 columnas: ['desc_provincia', 'cnt_ro_debajo_umbral', 'mto_fact_declarado_sunat', 'flg_activo_pep', 'desc_departamento', 'cod_ubigeo_cd', 'cod_sectorista_id', 'cod_ciiu_v4']
✓ Preprocessing aplicado
  Shape: (172506, 67)
  Dtypes:
key_value                    string[python]
cod_cli                      string[python]
codmes_lag1                  string[python]
cod_mes                               Int32
fec_constitucion                     object
mto_pas_soles                       float64
imp_trx_abonosefect_6m              float64
imp_trx_cargosefe_6m                float64
avg_trx_cargostot_3m                float64
max_trx_abonos_3m                   float64
cnt_trx_cargostot_3m                  Int32
cnt_trx_abonospromtot_3m            float64
rat_trx_abonosefectot_1m            float64
rat_trx_abonosefectot_3m            float64
rat_trx_abonosefectot_9m            float64
rat_mntcrgsefetot_1m                float64
num_edad_constitucion                 Int32
num_ant

,key_value,cod_cli,codmes_lag1,cod_mes,fec_constitucion,mto_pas_soles,imp_trx_abonosefect_6m,imp_trx_cargosefe_6m,avg_trx_cargostot_3m,max_trx_abonos_3m,...,alertas_por_antiguedad,ingresos_vs_facturacion,pasivo_vs_ingresos,ratio_egresos_exterior,ratio_ingresos_exterior,gap_riesgo_pep_lsb,tipo_alerta_n2,trx_riesgo_cliente,target_m,periodo_alerta
0,C996D04B0C345F298F48A33478E11DF6E78ED1D3A4962B...,0021536249,202603,202604,2025-02-24,3075.2500,5400.0,0.0,0.00,5000.00,...,0.0,0.000000,0.335726,0.0,0.0,0.0,0,0,0,0
1,A84E0854D16BC29F4A3055388D9E42BD78EB52277F9626...,0016085397,202603,202604,2001-04-11,532.7900,2869.0,75000.0,0.33,766.93,...,0.0,0.003424,0.023796,0.0,0.0,0.0,0,0,0,0
2,1CC46BFA8218FEB5883562FDA664E8FB8D4E410F3046E2...,0020742181,202603,202604,2022-05-27,0.0000,0.0,0.0,0.00,3300.00,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0,0,0,0
3,61D9981A8AB8587D6535C30FB44F1793ABCED78F5199C9...,0020353274,202603,202604,2023-05-11,41386.5253,6800.0,0.0,0.00,94195.25,...,0.0,0.000000,0.112694,0.0,0.0,0.0,0,0,0,0
4,070C26E8A5A4FCDF5050AA22F336AC5F86EE5E0F13C174...,0015509676,202603,202604,2016-10-12,0.0000,0.0,0.0,0.00,0.00,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0,0,0,0


In [8]:
# Convertir cod_cli al mismo tipo en ambos DataFrames
df_inference['key_value'] = df_inference['key_value'].astype('str')
df_inferencia_processed['key_value'] = df_inferencia_processed['key_value'].astype('str')

# Cruzar ambos DataFrames por cod_cli
df_merge = df_inference.merge(
    df_inferencia_processed,
    on='key_value',
    how='inner',
    suffixes=('_orig', '_proc')
)

print(f"✓ DataFrames cruzados por cod_cli")
print(f"  Registros coincidentes: {df_merge.shape[0]}")
print(f"  Columnas en merge: {df_merge.shape[1]}")

# Identificar columnas que existen en ambos DataFrames (sin sufijo)
cols_orig = set(df_inference.columns)
cols_proc = set(df_inferencia_processed.columns)
cols_comun = cols_orig & cols_proc

print(f"\n✓ Columnas comunes: {len(cols_comun)}")

# Comparar cada columna
diferencias = []
for col in sorted(cols_comun):
    if col == 'cod_cli':
        continue
    
    col_orig = f"{col}_orig"
    col_proc = f"{col}_proc"
    
    if col_orig not in df_merge.columns or col_proc not in df_merge.columns:
        continue
    
    # Comparar valores (usar fillna para manejar NaNs)
    no_coinciden = ((df_merge[col_orig].fillna(-999) != df_merge[col_proc].fillna(-999)).sum())
    
    if no_coinciden > 0:
        diferencias.append({
            'columna': col,
            'registros_diferentes': no_coinciden,
            'pct_diferencia': round(100 * no_coinciden / df_merge.shape[0], 2)
        })

# Mostrar resumen de diferencias
if diferencias:
    print(f"\n⚠ Se encontraron diferencias en {len(diferencias)} columnas:")
    df_diff_summary = pd.DataFrame(diferencias).sort_values('registros_diferentes', ascending=False)
    display(df_diff_summary)
    
    # Mostrar ejemplos de TODAS las columnas con diferencias
    print(f"\n📊 Ejemplos de diferencias:")
    for i, row in df_diff_summary.iterrows():
        col = row['columna']
        col_orig = f"{col}_orig"
        col_proc = f"{col}_proc"
        
        print(f"\n  → {col}: ({row['registros_diferentes']} registros diferentes)")
        mask_diff = (df_merge[col_orig].fillna(-999) != df_merge[col_proc].fillna(-999))
        sample = df_merge[mask_diff][[col_orig, col_proc]].head(5)
        display(sample)
else:
    print(f"\n✓ No se encontraron diferencias. Ambos DataFrames son idénticos.")

✓ DataFrames cruzados por cod_cli
  Registros coincidentes: 172505
  Columnas en merge: 116

✓ Columnas comunes: 48

⚠ Se encontraron diferencias en 31 columnas:


,columna,registros_diferentes,pct_diferencia
29,tipo_alerta_n2,172312,99.89
30,trx_riesgo_cliente,172291,99.88
7,cod_sectorista_id,156309,90.61
19,pasivo_vs_ingresos,109561,63.51
22,ratio_abonos_1m_vs_6m,91826,53.23
23,ratio_cargos_1m_vs_6m,40231,23.32
17,mto_pas_soles,27416,15.89
11,ingresos_vs_facturacion,27148,15.74
27,share_cp_egresos,6659,3.86
28,share_cp_ingresos,6638,3.85



📊 Ejemplos de diferencias:

  → tipo_alerta_n2: (172312 registros diferentes)


,tipo_alerta_n2_orig,tipo_alerta_n2_proc
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0



  → trx_riesgo_cliente: (172291 registros diferentes)


...

,rat_trx_abonosefectot_9m_orig,rat_trx_abonosefectot_9m_proc
8139,0.0,0.0053



  → num_antiguedad: (1 registros diferentes)


,num_antiguedad_orig,num_antiguedad_proc
8139,1.0,0



  → rat_trx_abonosefectot_3m: (1 registros diferentes)


,rat_trx_abonosefectot_3m_orig,rat_trx_abonosefectot_3m_proc
8139,0.0,0.01
